In [1]:
!pip install -q "RAGatouille[langchain]==0.0.9.post2" "langchain<0.2" "langchain-core<0.2" torch gdown

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# import gdown
# folder_url = "https://drive.google.com/drive/folders/1VnCQWO83fczOS6tnk4Ce8OeC-TG0O1Rs?usp=sharing"
# output_dir = "ragatouille_data"
# gdown.download_folder(folder_url, output=output_dir, quiet=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import time
import numpy as np
import random
import math
import json
import os
from tqdm import tqdm
import gc
from ragatouille import RAGPretrainedModel

/tmp/ipython-input-4245892356.py:9: UserWarning: 
********************************************************************************
RAGatouille WARNING: Future Release Notice
--------------------------------------------
RAGatouille version 0.0.10 will be migrating to a PyLate backend 
instead of the current Stanford ColBERT backend.
PyLate is a fully mature, feature-equivalent backend, that greatly facilitates compatibility.
However, please pin version <0.0.10 if you require the Stanford ColBERT backend.
********************************************************************************
  from ragatouille import RAGPretrainedModel


In [3]:
base_path = "/content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val"
#base_path = output_dir
def load_jsonl(path):
    data = []
    print(f"Attempting to load JSONL file: {path}") # Thêm log
    try:
        with open(path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                try:
                    # Loại bỏ khoảng trắng thừa ở đầu/cuối dòng
                    processed_line = line.strip()
                    if processed_line: # Đảm bảo dòng không rỗng sau khi strip
                        data.append(json.loads(processed_line))
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON on line {i+1} in {path}: {e}")
                    print(f"Problematic line content (first 100 chars): {line[:100]}...")
                    # Tùy chọn: Bỏ qua dòng lỗi hoặc dừng hẳn
                    # raise e # Dừng hẳn nếu muốn biết lỗi ngay lập tức
                    print("Skipping this problematic line.") # Bỏ qua dòng lỗi và tiếp tục
    except FileNotFoundError:
        print(f"ERROR: File not found at {path}")
        raise
    except Exception as e:
        print(f"An unexpected error occurred while loading {path}: {e}")
        raise
    print(f"Successfully loaded {len(data)} records from {path}")
    return data

def load_qrels(path):
    qrels = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            qid, _, docid, rel = line.strip().split()
            qrels.setdefault(qid, {})[docid] = int(rel)
    return qrels

In [4]:
def save_jsonl(items, path):
    with open(path, "w", encoding="utf-8") as f:
        for d in items:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

def save_qrels(qrels_list, path):
    # qrels_list là list của (query_id, doc_id, rel)
    with open(path, "w", encoding="utf-8") as f:
        # mỗi dòng: query_id \t 0 \t doc_id \t relevance
        for qid, docid, rel in qrels_list:
            f.write(f"{qid}\t0\t{docid}\t{rel}\n")

def convert_ms_marco(split="validation", out_dir=os.path.join(base_path, "data/ms_marco_val")):
    ds = load_dataset("microsoft/ms_marco", 'v2.1', split=split)
    os.makedirs(out_dir, exist_ok=True)

    # Queries
    queries = [{"id": str(rec["query_id"]), "text": rec["query"]} for rec in ds]

    # Corpus & Qrels
    corpus_map = {}
    qrels = []

    for rec in ds:
        qid = str(rec["query_id"])
        passages = rec["passages"]
        texts = passages["passage_text"]
        selected = passages["is_selected"]

        for idx, text in enumerate(texts):
            if not text:
                continue
            doc_id = f"{qid}_{idx}"
            corpus_map[doc_id] = text
            if selected[idx] == 1:
                qrels.append((qid, doc_id, 1))

    corpus = [{"id": did, "text": txt} for did, txt in corpus_map.items()]

    # Save
    save_jsonl(corpus, os.path.join(out_dir, "corpus.jsonl"))
    save_jsonl(queries, os.path.join(out_dir, "queries.jsonl"))
    save_qrels(qrels, os.path.join(out_dir, "qrels.tsv"))

    print("Done convert MS MARCO split:", split)

# if exists the folder, skip
if not os.path.exists(os.path.join(base_path, "corpus.jsonl")):
    convert_ms_marco(split="validation", out_dir=base_path)

# if not os.path.exists(os.path.join(base_path, "data/ms_marco_train/corpus.jsonl")):
#     convert_ms_marco(split="train", out_dir=base_path)


In [5]:
class RagatouilleRetriever:
    def __init__(self, collection_texts, collection_ids, index_root,
                 checkpoint='colbert-ir/colbertv2.0',
                 nbits=2, doc_maxlen=100, overwrite=False): # <-- Added 'overwrite'

        self.index_root = index_root
        self.checkpoint = checkpoint
        # RAGatouille manages index names and paths internally
        self.index_name = f'colbert_index_full_{nbits}bits'

        print(f"Initializing RAGatouille with checkpoint: {checkpoint}")
        self.retriever = RAGPretrainedModel.from_pretrained(
            checkpoint,
            index_root=self.index_root,
            n_gpu=-1
        )


    def index_collection(self, collection_texts, collection_ids, doc_maxlen, overwrite):
        """Tạo Index cho collection."""

        # The manual os.path.exists() check is REMOVED.
        # We let RAGatouille handle index checking via 'overwrite_index'.

        print(f"Indexing full collection... (Index: {self.index_name}, Overwrite: {overwrite})")

        self.retriever.index(
            collection=collection_texts,
            document_ids=collection_ids,
            index_name=self.index_name,
            max_document_length=doc_maxlen,
            overwrite_index=overwrite,
            use_faiss=True
        )
        print("Indexing complete.")

    def search_batch(self, queries, k=3, batch_size=64):
        """
        Thực hiện tìm kiếm ColBERT theo lô (batch).
        RAGatouille's .search() method natively supports batch queries!
        """
        print(f"RAGatouille searching {len(queries)} queries (k={k})...")

        # Pass the list of query strings directly to RAGatouille
        # It handles batching internally (we can ignore the batch_size param)
        batch_results = self.retriever.search(
            query=queries,
            k=k,
            index_name=self.index_name
        )

        all_results_tuples = []

        # Process results into the (doc_id, score) format expected by 'evaluate'
        for q_results in tqdm(batch_results, desc="Mapping results"):
            # q_results is a list of dicts for a single query
            # e.g., [{'content': '...', 'score': 0.5, 'document_id': '...'}, ...]
            current_q_results = [
                (res['document_id'], res['score']) for res in q_results
            ]
            all_results_tuples.append(current_q_results)

        return all_results_tuples

In [6]:
import math
import json
import numpy as np
from tqdm import tqdm

def recall_at_k(run, qrels, k):
    # run: list of (docid, score)
    relevant = {d for d, rel in qrels.items() if rel > 0}
    retrieved = {d for d, _ in run[:k]}
    if not relevant:
        return 0.0
    return len(relevant & retrieved) / len(relevant)

def dcg_at_k(run, qrels, k):
    import math
    return sum(
        (2**qrels.get(doc, 0) - 1) / math.log2(idx + 2)
        for idx, (doc, _) in enumerate(run[:k])
    )

def ndcg_at_k(run, qrels, k):
    ideal = sorted(qrels.values(), reverse=True)[:k]
    idcg = sum((2**rel - 1) / math.log2(idx + 2) for idx, rel in enumerate(ideal))
    if idcg == 0:
        return 0.0
    return dcg_at_k(run, qrels, k) / idcg

def normalize_scores(scores_dict):
    if not scores_dict: return {}
    vals = list(scores_dict.values())
    min_v, max_v = min(vals), max(vals)
    range_v = max_v - min_v
    if range_v == 0: # Tránh chia cho 0 nếu tất cả điểm số bằng nhau
        return {k: 0.5 for k in scores_dict} # Hoặc trả về 0 hoặc 1 tùy logic
    return {k: (v - min_v) / range_v for k, v in scores_dict.items()}

In [7]:
# @title Cell 17: Cập nhật hàm evaluate (LƯU JSONL)
def evaluate(retriever, queries, qrels, k=10, retrieval_k=50, batch_size=64, save_path=None): # <<< THÊM MỚI save_path
    """Đánh giá retriever trên tập queries đầy đủ đã chọn ngẫu nhiên."""

    q_texts = [q["text"] for q in queries]
    q_ids = [q["id"] for q in queries]
    eval_name = retriever.__class__.__name__

    all_final_outputs = [] # <<< THÊM MỚI

    print(f"--- Starting Evaluation for: {eval_name} (Queries: {len(queries)}, k={k}) ---")

    start_time = time.time()

    current_retrieval_k = retrieval_k
    all_final_runs_tuples = retriever.search_batch(q_texts, k=current_retrieval_k, batch_size=batch_size)

    end_time = time.time()
    total_time = end_time - start_time
    print(f"Retrieval completed in {total_time:.2f} seconds.")

    ndcgs, recalls = [], []

    for i in range(len(queries)):
        qid = q_ids[i]
        final_run_tuples = all_final_runs_tuples[i][:k] # Cắt top k
        qrel_for_query = qrels.get(qid, {})

        ndcgs.append(ndcg_at_k(final_run_tuples, qrel_for_query, k))
        recalls.append(recall_at_k(final_run_tuples, qrel_for_query, k))

        full_results_list = [{"doc_id": docid, "score": float(score)}
                             for docid, score in all_final_runs_tuples[i]]

        all_final_outputs.append({
            "query_id": qid,
            "query_text": q_texts[i],
            "retriever_info": eval_name,
            "results": full_results_list
        })

    if save_path:
        # Tạo thư mục nếu chưa tồn tại
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        print(f"Saving {len(all_final_outputs)} final results to {save_path}...")
        try:
            with open(save_path, "w", encoding="utf-8") as f:
                for record in all_final_outputs:
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
            print("Saved successfully.")
        except Exception as e:
            print(f"Error saving results to {save_path}: {e}")

    mean_ndcg = float(np.mean(ndcgs)) if ndcgs else 0.0
    mean_recall = float(np.mean(recalls)) if recalls else 0.0

    print(f"--- Final Metrics for {eval_name} ---")
    print(f"Average nDCG@{k}: {mean_ndcg:.4f}")
    print(f"Average Recall@{k}: {mean_recall:.4f}")
    print("." * 30)

    return mean_ndcg, mean_recall, all_final_outputs

In [8]:
NUM_CORPUS_TARGET = 50000 # Kích thước Corpus mục tiêu
NUM_QUERY = 1000          # Số lượng query cần đánh giá

# --- 1. Tải Dữ liệu Gốc ---
corpus = load_jsonl(os.path.join(base_path, "corpus.jsonl"))
queries_full = load_jsonl(os.path.join(base_path, "queries.jsonl"))
qrels_full = load_qrels(os.path.join(base_path, "qrels.tsv"))

print(f"Original Corpus size: {len(corpus):,}")
print(f"Original Queries size: {len(queries_full):,}")

# --- 2. Chọn 1000 Queries và Qrels tương ứng ---
random.seed(42)
if len(queries_full) > NUM_QUERY:
    queries_eval = random.sample(queries_full, NUM_QUERY)
else:
    queries_eval = queries_full # Nếu ít hơn 1000 thì dùng hết

qids_eval = {q['id'] for q in queries_eval}

# Lọc Qrels để chỉ chứa các entries cho 1000 queries đã chọn
qrels_eval = {qid: docs for qid, docs in qrels_full.items() if qid in qids_eval}

# Lấy tất cả Document IDs relevant (ground truth)
relevant_doc_ids = set()
for docs in qrels_eval.values():
    relevant_doc_ids.update({docid for docid, rel in docs.items() if rel > 0})

# Loại bỏ các query không còn relevant doc nào sau khi lọc (thực tế ít xảy ra)
qids_with_relevant_doc = {qid for qid, docs in qrels_eval.items() if docs}
queries_eval = [q for q in queries_eval if q['id'] in qids_with_relevant_doc]

# --- 3. Chọn Noise Documents để đạt 50,000 Corpus ---
corpus_map_full = {doc['id']: doc for doc in corpus}
all_doc_ids = set(corpus_map_full.keys())
noise_candidate_ids = all_doc_ids - relevant_doc_ids

num_relevant_docs = len(relevant_doc_ids)
required_noise = max(0, NUM_CORPUS_TARGET - num_relevant_docs)

random.seed(42) # Giữ seed để random noise
# Đảm bảo không lấy nhiều hơn số lượng có sẵn
num_noise_to_take = min(required_noise, len(noise_candidate_ids))

if num_noise_to_take > 0:
    noise_doc_ids = random.sample(list(noise_candidate_ids), num_noise_to_take)
else:
    noise_doc_ids = []

# --- 4. Xây dựng Corpus và Qrels Cuối cùng ---
final_corpus_ids = relevant_doc_ids.union(set(noise_doc_ids))

# Lọc corpus cuối cùng
corpus_subset = [corpus_map_full[doc_id] for doc_id in final_corpus_ids]

# Cập nhật các biến cho bước tiếp theo
queries = queries_eval
qrels = qrels_eval
corpus = corpus_subset

# Dữ liệu cần thiết cho việc index Milvus
corpus_texts = [doc['text'] for doc in corpus]
corpus_ids = [doc['id'] for doc in corpus]
corpus_map = {doc['id']: doc['text'] for doc in corpus} # Cần cập nhật corpus_map

print("-" * 50)
print(f"Final Corpus size: {len(corpus):,} documents (Relevant: {num_relevant_docs:,}, Noise: {len(noise_doc_ids):,})")
print(f"Final Queries size: {len(queries):,} queries (Tất cả đều có relevant doc trong corpus)")
print(f"Final Qrels size: {sum(len(v) for v in qrels.values()):,} relevant entries")
print("-" * 50)

results_dir = os.path.join(base_path, "results_optimized_2")
os.makedirs(results_dir, exist_ok=True)

# NUM_QUERY=1000

# # sử dụng seed 42 chọn ra 200 query
# import random
# random.seed(42)
# queries = random.sample(queries, NUM_QUERY)

# results_dir = os.path.join(base_path, "results_optimized")
# os.makedirs(results_dir, exist_ok=True)

corpus_texts = [doc['text'] for doc in corpus]
corpus_ids = [doc['id'] for doc in corpus]

Attempting to load JSONL file: /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/corpus.jsonl
Successfully loaded 1008985 records from /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/corpus.jsonl
Attempting to load JSONL file: /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/queries.jsonl
Successfully loaded 101093 records from /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/queries.jsonl
Original Corpus size: 1,008,985
Original Queries size: 101,093
--------------------------------------------------
Final Corpus size: 50,000 documents (Relevant: 595, Noise: 49,405)
Final Queries size: 556 queries (Tất cả đều có relevant doc trong corpus)
Final Qrels size: 595 relevant entries
--------------------------------------------------


In [9]:
import sys; sys.path.insert(0, 'ColBERT/')
from colbert import Indexer, Searcher
from colbert.infra import Run, RunConfig, ColBERTConfig
from colbert.data import Queries, Collection

In [10]:
 #@title Cell 10: Run Evaluation with RAGatouille (Corrected)
# This is the updated main execution block.
# It now calls 'RagatouilleRetriever'.

results = {} # Initialize results

print("\n" + "="*50)
print("BẮT ĐẦU ĐÁNH GIÁ RAGATOUILLE (COLBERTV2)")
print(f"Tổng số passages được index: {len(corpus_texts):,}")
print("="*50)

# Khởi tạo RagatouilleRetriever
colbert_retriever = RagatouilleRetriever(
    collection_texts=corpus_texts,
    collection_ids=corpus_ids, # Pass the IDs
    index_root=os.path.join(base_path, "ragatouille_data"), # Use a new root for RAGatouille indexes
    checkpoint='colbert-ir/colbertv2.0',
    nbits=2,
    doc_maxlen=80,
    overwrite=True
)


BẮT ĐẦU ĐÁNH GIÁ RAGATOUILLE (COLBERTV2)
Tổng số passages được index: 50,000
Initializing RAGatouille with checkpoint: colbert-ir/colbertv2.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


artifact.metadata: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


In [11]:
colbert_retriever.index_collection(
    collection_texts=corpus_texts,
    collection_ids=corpus_ids,
    doc_maxlen=80,
    overwrite=True
)

Indexing full collection... (Index: colbert_index_full_2bits, Overwrite: True)
________________________________________________________________________________
WARNING! You have a GPU available, but only `faiss-cpu` is currently installed.
 This means that indexing will be slow. To make use of your GPU.
Please install `faiss-gpu` by running:
pip uninstall --y faiss-cpu & pip install faiss-gpu
 ________________________________________________________________________________
Will continue with CPU indexing in 5 seconds...


[Nov 01, 08:35:40] #> Creating directory /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/ragatouille_data/colbert/indexes/colbert_index_full_2bits 


[Nov 01, 08:35:44] [0] 		 #> Encoding 41933 passages..


/usr/local/lib/python3.12/dist-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()


[Nov 01, 08:36:51] [0] 		 avg_doclen_est = 56.98283004760742 	 len(local_sample) = 41,933
[Nov 01, 08:36:55] [0] 		 Creating 16,384 partitions.
[Nov 01, 08:36:55] [0] 		 *Estimated* 3,261,526 embeddings.
[Nov 01, 08:36:55] [0] 		 #> Saving the indexing plan to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/ragatouille_data/colbert/indexes/colbert_index_full_2bits/plan.json ..
[Nov 01, 09:06:09] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


W1101 09:06:09.160000 1401 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W1101 09:06:09.160000 1401 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.


[Nov 01, 09:07:29] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


...

3it [01:38, 32.85s/it]
100%|██████████| 3/3 [00:00<00:00, 101.22it/s]


[Nov 01, 09:10:26] #> Optimizing IVF to store map from centroids to list of pids..
[Nov 01, 09:10:26] #> Building the emb2pid mapping..
[Nov 01, 09:10:26] len(emb2pid) = 3256236


100%|██████████| 16384/16384 [00:00<00:00, 48304.97it/s]

[Nov 01, 09:10:27] #> Saved optimized IVF to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/ragatouille_data/colbert/indexes/colbert_index_full_2bits/ivf.pid.pt


Done indexing!
Indexing complete.


In [13]:
# Sửa lỗi: Thêm biến 'colbert_outputs' để nhận giá trị thứ 3
colbert_ndcg, colbert_recall, colbert_outputs = evaluate(
    retriever=colbert_retriever,
    queries=queries,
    qrels=qrels,
    k=3,
    retrieval_k=30,
    batch_size=128,
    # Sửa lỗi: Cung cấp một tên file cụ thể (ví dụ: "eval_results.jsonl")
    # thay vì chỉ truyền vào một thư mục
    save_path=os.path.join(results_dir, "colbert_evaluation_results.jsonl")
)

results['RAGatouille_ColBERTv2'] = {'nDCG@3': colbert_ndcg, 'Recall@3': colbert_recall}

print("\n" + "="*50)
print("TÓM TẮT KẾT QUẢ CUỐI CÙNG CHO RAGATOUILLE:")
print(json.dumps(results, indent=4))
print("="*50)

with open(os.path.join(results_dir, 'results_ragatouille.json'), 'w') as f:
    json.dump(results, f, indent=4)

--- Starting Evaluation for: RagatouilleRetriever (Queries: 556, k=3) ---
RAGatouille searching 556 queries (k=30)...


556it [00:06, 91.98it/s] 
Mapping results: 100%|██████████| 556/556 [00:00<00:00, 103839.75it/s]

Retrieval completed in 6.49 seconds.
Saving 556 final results to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/results_optimized_2/colbert_evaluation_results.jsonl...
Saved successfully.
--- Final Metrics for RagatouilleRetriever ---
Average nDCG@3: 0.9638
Average Recall@3: 0.9412
..............................

TÓM TẮT KẾT QUẢ CUỐI CÙNG CHO RAGATOUILLE:
{
    "RAGatouille_ColBERTv2": {
        "nDCG@3": 0.9638485656689099,
        "Recall@3": 0.9412470023980816
    }
}
